# GwenLand glcuda Wave 113 — repaired Q8 head-to-head on Tesla T4

Thirty sessions: ten retained GwenLand, ten Wave111, ten pinned llama.cpp.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
import statistics
import time
from pathlib import Path
import shutil
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave113-residual-fusion-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "bd5c956bafb3bb6738c3f1de348a4ebb55d9c29f"
SOURCE_REV = "a9ca6070acabd4d6ff677532716b1e4bf92a7c98"
PATCH_SHA256 = "e8019acc8810445c9e29ad32eb60e7bd76c8607c6d2d0df8e9f1fea68911f8d3"
PATCH_GZIP_B64 = """..."""
ROOT = Path("/kaggle/working/wave113")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave113-h2h-q8-results.zip")
HF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
HF_REVISION = "9217f5db79a29953eb74d5343926648285ec7e67"
HF_FILENAME = "qwen2.5-0.5b-instruct-q8_0.gguf"
HF_EXPECTED_BYTES = 675710816
HF_EXPECTED_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
COLD_ITERS = 1
WARMUP_ITERS = 5
MEASURE_ITERS = 10
PRODUCTION_REPEATS = 10

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)


def run(cmd, *, cwd=None, env=None, timeout=7200, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run(
        [str(x) for x in cmd], cwd=cwd, env=merged, text=True,
        capture_output=True, timeout=timeout,
    )
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p


def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )


def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = hashlib.sha256(FINAL_ZIP.read_bytes()).hexdigest()
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest


def fail(phase):
    (RESULTS / "FAILED.json").write_text(
        json.dumps({"phase": phase, "traceback": traceback.format_exc()}, indent=2),
        encoding="utf-8",
    )
    archive()
    raise RuntimeError(f"Wave 113 failed in {phase}")


phase = "bootstrap"
try:
    patch = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    got = hashlib.sha256(patch).hexdigest()
    if got != PATCH_SHA256:
        raise RuntimeError(f"patch hash mismatch: {got} != {PATCH_SHA256}")
    patch_path = RESULTS / "wave113.patch"
    patch_path.write_bytes(patch)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(patch),
    }, indent=2), encoding="utf-8")

    gpu = run([
        "nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
        "--format=csv,noheader,nounits",
    ], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"Wave 113 requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff_check = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff_check)

    phase = "build-test"
    cargo_candidates = [
        shutil.which("cargo"), Path.home() / ".cargo/bin/cargo",
        "/usr/local/cargo/bin/cargo", "/opt/rust/bin/cargo",
        "/opt/conda/bin/cargo", "/usr/local/bin/cargo", "/usr/bin/cargo",
    ]
    cargo = next((str(path) for path in cargo_candidates if path and Path(path).is_file()), None)
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_script = ROOT / "rustup-init.sh"
        with urllib.request.urlopen("https://sh.rustup.rs", timeout=120) as response:
            rustup_script.write_bytes(response.read())
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {"CARGO_HOME": cargo_home, "RUSTUP_HOME": rustup_home}
        install = run([
            "bash", rustup_script, "-y", "--profile", "minimal",
            "--default-toolchain", "stable", "--no-modify-path",
        ], env=cargo_env, timeout=1800)
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after discovery/bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo, "bootstrapped": bootstrapped,
        "candidates": [str(path) for path in cargo_candidates if path],
    }, indent=2), encoding="utf-8")
    common = {**cargo_env, "CARGO_TARGET_DIR": TARGET, "CUDA_VISIBLE_DEVICES": "0"}
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    summary = next((x for x in (tests.stdout + tests.stderr).splitlines()
                    if x.startswith("test result:")), "")
    if "66 passed" not in summary or "0 failed" not in summary:
        raise RuntimeError(f"unexpected host test summary: {summary}")
    parity = run([
        cargo, "test", "--release", "-p", "glcuda", "--test", "parity",
        "--locked", "--", "--nocapture", "--test-threads=1",
    ], cwd=TREE, env=common, timeout=14400, check=False)
    save("cargo-cuda-parity.log", parity)
    parity_summary = next((x for x in (parity.stdout + parity.stderr).splitlines()
                           if x.startswith("test result:")), "")
    if parity.returncode or "0 failed" not in parity_summary:
        raise RuntimeError(f"full CUDA parity failed: {parity_summary}")

    build = run([
        cargo, "build", "--release", "-p", "glbench", "--locked",
    ], cwd=TREE, env=common, timeout=7200, check=False)
    save("cargo-build-glbench.log", build)
    if build.returncode:
        raise RuntimeError("Wave 113 glbench release build failed")
    CARGO = cargo
    CARGO_ENV = cargo_env
    GLBENCH = TARGET / "release/glbench"
    PARITY = {"summary": parity_summary, "returncode": parity.returncode}
    BOOTSTRAP_OK = True
    print("WAVE113_BOOTSTRAP_OK", json.dumps(PARITY), flush=True)
except Exception:
    fail(phase)


## Fetch and verify the exact Q8_0 model


In [ ]:
if not globals().get("BOOTSTRAP_OK"):
    raise RuntimeError("Bootstrap gate did not pass")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch_pinned_model():
    model = ROOT / HF_FILENAME
    part = ROOT / f"{HF_FILENAME}.part"
    if model.is_file() and model.stat().st_size == HF_EXPECTED_BYTES and sha256_file(model) == HF_EXPECTED_SHA256:
        return model
    if model.exists():
        model.unlink()
    if part.exists():
        part_size = part.stat().st_size
        if part_size == HF_EXPECTED_BYTES:
            if sha256_file(part) == HF_EXPECTED_SHA256:
                part.replace(model)
                return model
            part.unlink()
        elif part_size > HF_EXPECTED_BYTES:
            part.unlink()
    url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true"
    for attempt in range(1, 6):
        start = part.stat().st_size if part.exists() else 0
        headers = {"User-Agent": "GwenLand-glcuda-Wave113-Q8-H2H/1.0", "Accept-Encoding": "identity"}
        if start:
            headers["Range"] = f"bytes={start}-"
        request = urllib.request.Request(url, headers=headers)
        try:
            response = urllib.request.urlopen(request, timeout=120)
            status = getattr(response, "status", response.getcode())
            if start and status != 206:
                response.close()
                part.unlink(missing_ok=True)
                start = 0
                response = urllib.request.urlopen(
                    urllib.request.Request(url, headers={"User-Agent": "GwenLand-glcuda-Wave113-Q8-H2H/1.0", "Accept-Encoding": "identity"}),
                    timeout=120,
                )
                status = getattr(response, "status", response.getcode())
            if status not in (200, 206):
                raise RuntimeError(f"HTTP {status}")
            mode = "ab" if start and status == 206 else "wb"
            downloaded = start
            last_print = time.monotonic()
            with response, part.open(mode) as output:
                while True:
                    chunk = response.read(8 << 20)
                    if not chunk:
                        break
                    output.write(chunk)
                    downloaded += len(chunk)
                    if time.monotonic() - last_print >= 20:
                        print(f"model fetch {downloaded / (1 << 20):.1f}/{HF_EXPECTED_BYTES / (1 << 20):.1f} MiB")
                        last_print = time.monotonic()
            if part.stat().st_size != HF_EXPECTED_BYTES:
                raise RuntimeError(f"truncated model: {part.stat().st_size}/{HF_EXPECTED_BYTES}")
            digest = sha256_file(part)
            if digest != HF_EXPECTED_SHA256:
                part.unlink(missing_ok=True)
                raise RuntimeError(f"model SHA mismatch: {digest}")
            part.replace(model)
            return model
        except Exception as exc:
            print(f"fetch attempt {attempt}/5 failed: {exc}")
            if attempt == 5:
                raise
            time.sleep(min(30, 2 ** attempt))

try:
    MODEL_PATH = fetch_pinned_model()
    MODEL_META = {"repo": HF_REPO, "revision": HF_REVISION, "filename": HF_FILENAME, "bytes": MODEL_PATH.stat().st_size, "sha256": sha256_file(MODEL_PATH)}
    (RESULTS / "model.json").write_text(json.dumps(MODEL_META, indent=2), encoding="utf-8")
    MODEL_OK = True
    print(json.dumps(MODEL_META, indent=2))
except Exception:
    fail("model-fetch")

## Build pinned stock-CUDA llama.cpp


In [ ]:
if not globals().get("MODEL_OK"):
    raise RuntimeError("Model gate did not pass")

import glob

LLAMA_REPO = "https://github.com/ggml-org/llama.cpp.git"
LLAMA_PIN = "4d9176092d00586775af140581bb0b558ddc4389"
LLAMA_DIR = ROOT / "llama.cpp-wave113"
CUDA_ARCH = "75-real"
BUILD_JOBS = min(2, os.cpu_count() or 1)
LLAMA_BUILD_FLAGS = [
    "GGML_CUDA=ON", "CMAKE_BUILD_TYPE=Release",
    f"CMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}", "LLAMA_CURL=OFF",
    "LLAMA_BUILD_TESTS=OFF", "LLAMA_BUILD_EXAMPLES=OFF",
    "LLAMA_BUILD_TOOLS=ON", "LLAMA_BUILD_SERVER=ON", "LLAMA_BUILD_APP=OFF",
    "GGML_CUDA_NCCL=OFF",
]

def sh(cmd, cwd=None, timeout=7200, env=None):
    merged = os.environ.copy()
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    try:
        p = subprocess.run(
            [str(x) for x in cmd], cwd=str(cwd) if cwd else None,
            env=merged, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
            stderr=subprocess.PIPE, text=True, errors="replace", timeout=timeout,
        )
        return p.returncode, p.stdout, p.stderr
    except subprocess.TimeoutExpired as exc:
        return 124, exc.stdout or "", (exc.stderr or "") + "\nTIMEOUT"

def save_command_log(name, cmd, rc, out, err):
    (RESULTS / name).write_text(
        f"$ {' '.join(str(x) for x in cmd)}\nreturncode={rc}\n\n"
        f"STDOUT\n{out}\n\nSTDERR\n{err}",
        encoding="utf-8",
    )

def fail_with_archive(phase, message):
    (RESULTS / "FAILED.json").write_text(
        json.dumps({"phase": phase, "message": message}, indent=2),
        encoding="utf-8",
    )
    archive()
    raise RuntimeError(f"{message}; partial archive: {FINAL_ZIP}")

if LLAMA_DIR.exists():
    shutil.rmtree(LLAMA_DIR)
clone_cmd = ["git", "clone", "--filter=blob:none", "--no-checkout", LLAMA_REPO, LLAMA_DIR]
rc, out, err = sh(clone_cmd, timeout=1800)
save_command_log("llama-git-clone.log", clone_cmd, rc, out, err)
if rc:
    fail_with_archive("llama-clone", "llama.cpp clone failed")

fetch_cmd = ["git", "fetch", "--depth", "1", "origin", LLAMA_PIN]
rc, out, err = sh(fetch_cmd, cwd=LLAMA_DIR, timeout=1800)
save_command_log("llama-git-fetch.log", fetch_cmd, rc, out, err)
if rc:
    fail_with_archive("llama-pin", f"cannot fetch pinned llama.cpp commit {LLAMA_PIN}")
checkout_cmd = ["git", "checkout", "--detach", LLAMA_PIN]
rc, out, err = sh(checkout_cmd, cwd=LLAMA_DIR, timeout=300)
save_command_log("llama-git-checkout.log", checkout_cmd, rc, out, err)
if rc:
    fail_with_archive("llama-pin", f"cannot checkout pinned llama.cpp commit {LLAMA_PIN}")
rc, out, err = sh(["git", "rev-parse", "HEAD"], cwd=LLAMA_DIR, timeout=120)
LLAMA_COMMIT = out.strip()
if rc or LLAMA_COMMIT != LLAMA_PIN:
    fail_with_archive("llama-pin", f"llama.cpp pin drift: {LLAMA_COMMIT!r} != {LLAMA_PIN}")
print("llama.cpp pinned commit:", LLAMA_COMMIT)

# Kaggle's CUDA image has the runtime driver but CMake does not always discover
# CUDA::cuda_driver. Resolve the authoritative ldconfig path and pass it to
# CMake; unlike GGML_CUDA_NO_VMM this does not alter the measured engine.
libcuda_candidates = []
rc, out, err = sh(["ldconfig", "-p"], timeout=120)
for line in out.splitlines():
    if "libcuda.so" in line and "=>" in line:
        path = line.split("=>")[-1].strip()
        if os.path.exists(path):
            libcuda_candidates.append(path)
for pattern in (
    "/usr/local/nvidia/lib64/libcuda.so",
    "/usr/local/cuda*/lib64/stubs/libcuda.so",
    "/usr/local/cuda*/targets/*/lib/stubs/libcuda.so",
):
    libcuda_candidates.extend(glob.glob(pattern))
libcuda_candidates = sorted(
    set(libcuda_candidates),
    key=lambda p: (os.path.basename(p) != "libcuda.so", len(p)),
)
if not libcuda_candidates:
    fail_with_archive("llama-configure", "libcuda.so not found; refusing a NO_VMM benchmark")
LIBCUDA = libcuda_candidates[0]
print("libcuda:", LIBCUDA)

use_ccache = shutil.which("ccache") is not None
cmake_cmd = ["cmake", "-S", LLAMA_DIR, "-B", LLAMA_DIR / "build"]
cmake_cmd += [f"-D{flag}" for flag in LLAMA_BUILD_FLAGS]
cmake_cmd += [
    f"-DCUDA_cuda_driver_LIBRARY={LIBCUDA}",
    f"-DCUDA_CUDA_LIBRARY={LIBCUDA}",
    f"-DCMAKE_LIBRARY_PATH={os.path.dirname(LIBCUDA)}",
]
if use_ccache:
    cmake_cmd += [
        "-DCMAKE_C_COMPILER_LAUNCHER=ccache",
        "-DCMAKE_CXX_COMPILER_LAUNCHER=ccache",
        "-DCMAKE_CUDA_COMPILER_LAUNCHER=ccache",
    ]
rc, out, err = sh(cmake_cmd, timeout=1800)
save_command_log("llama-cmake-configure.log", cmake_cmd, rc, out, err)
if rc:
    fail_with_archive("llama-configure", "stock CUDA llama.cpp configure failed")

build_cmd = [
    "cmake", "--build", LLAMA_DIR / "build", "--config", "Release",
    "-j", str(BUILD_JOBS), "--target", "llama-bench", "llama-tokenize",
]
build_started = time.time()
rc, out, err = sh(build_cmd, timeout=21600)
LLAMA_BUILD_SECS = time.time() - build_started
save_command_log("llama-cmake-build.log", build_cmd, rc, out, err)
if rc:
    fail_with_archive("llama-build", "stock CUDA llama.cpp build failed")

LLAMA_BIN_DIR = LLAMA_DIR / "build/bin"
LLAMA_BENCH = LLAMA_BIN_DIR / "llama-bench"
LLAMA_TOKENIZE = LLAMA_BIN_DIR / "llama-tokenize"
if not LLAMA_BENCH.is_file() or not LLAMA_TOKENIZE.is_file():
    fail_with_archive("llama-build", "llama.cpp binaries missing after successful build")
print(f"llama.cpp build complete in {LLAMA_BUILD_SECS / 60:.1f} min")


## Reject CPU, multi-GPU, and revision drift


In [ ]:
LB_ENV = {"CUDA_VISIBLE_DEVICES": "0"}
smoke_cmd = [LLAMA_BENCH, "-m", MODEL_PATH, "-p", "8", "-n", "0",
             "-r", "1", "-ngl", "99", "-o", "json"]
rc, out, err = sh(smoke_cmd, timeout=900, env=LB_ENV)
save_command_log("llamabench-smoke.log", smoke_cmd, rc, out, err)
if rc:
    fail_with_archive("llama-smoke", "llama-bench smoke failed")
try:
    smoke_rows = json.loads(out)
except Exception as exc:
    fail_with_archive("llama-smoke", f"llama-bench smoke JSON failed: {exc}")
smoke = smoke_rows[-1]
cuda_markers = ("ggml_cuda_init", "CUDA0", "using CUDA", "found 1 CUDA")
if (smoke.get("backends") != "CUDA" or smoke.get("gpu_info") != "Tesla T4" or
        int(smoke.get("n_gpu_layers", 0)) != 99 or
        not any(marker in err for marker in cuda_markers) or
        "found 2 CUDA" in err):
    fail_with_archive("llama-smoke", f"llama.cpp single-T4 CUDA gate failed: {smoke}")
if not str(smoke.get("build_commit", "")).startswith(LLAMA_COMMIT[:7]):
    fail_with_archive("llama-smoke", f"llama.cpp binary commit drift: {smoke}")
LLAMA_OK = True
print(json.dumps(smoke, indent=2))


## Exact tokenizer gate and 30-session production comparison


In [ ]:
if not globals().get("MODEL_OK") or not globals().get("LLAMA_OK"):
    raise RuntimeError("model/llama gate did not pass")

prompt_unit = "Measure this deterministic systems prompt carefully. Explain how token-parallel integer matrix multiplication uses shared memory, Tensor Cores, and fixed launch geometry. "
FIXED_PROMPT = prompt_unit * 8
CHATML_PROMPT = (
    "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
    "<|im_start|>user\n" + FIXED_PROMPT + "<|im_end|>\n"
    "<|im_start|>assistant\n"
)
COMMON_ENV = {
    **CARGO_ENV,
    "CUDA_VISIBLE_DEVICES": "0",
    "CARGO_TARGET_DIR": str(TARGET),
    "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
    "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
    "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
    "GLCUDA_GEMM_N16_PREFETCH": "1",
    "GLCUDA_ATTN_MMA4": "1", "GLCUDA_ATTN_MMA4_REGQ": "1",
    "GLCUDA_ATTN_MMA4_AV": "1",
}
ARM_ENV = {
    "gwen_retained": {},
    "gwen_wave111": {"GLCUDA_DEFER_FFN_RESIDUAL": "1"},
}
ORDERS = [
    ["gwen_retained", "gwen_wave111", "llamacpp"],
    ["gwen_wave111", "llamacpp", "gwen_retained"],
    ["llamacpp", "gwen_retained", "gwen_wave111"],
    ["gwen_retained", "llamacpp", "gwen_wave111"],
    ["llamacpp", "gwen_wave111", "gwen_retained"],
    ["gwen_wave111", "gwen_retained", "llamacpp"],
    ["gwen_retained", "gwen_wave111", "llamacpp"],
    ["gwen_wave111", "llamacpp", "gwen_retained"],
    ["llamacpp", "gwen_retained", "gwen_wave111"],
    ["gwen_retained", "llamacpp", "gwen_wave111"],
]
assert len(ORDERS) == PRODUCTION_REPEATS
assert all(sum(order.count(arm) for order in ORDERS) == 10 for arm in (*ARM_ENV, "llamacpp"))

def percentile(values, q):
    values = sorted(values)
    index = (len(values) - 1) * q
    lo, hi = math.floor(index), math.ceil(index)
    return values[lo] if lo == hi else values[lo] * (hi - index) + values[hi] * (index - lo)

def json_lines(hay, prefix):
    return [json.loads(x) for x in re.findall(re.escape(prefix) + r"\s*(\{[^\n]+\})", hay)]

def last_json_line(hay, prefix):
    rows = json_lines(hay, prefix)
    if not rows:
        raise RuntimeError(f"dispatch line missing: {prefix}")
    return rows[-1]

def run_gwenland(arm, out, cold, warmup, iters):
    cmd = [
        GLBENCH, "run", "--engine", "glcuda", "--model", MODEL_PATH,
        "--prompt", FIXED_PROMPT, "--tokens", "1",
        "--cold-iters", str(cold), "--warmup", str(warmup), "--iters", str(iters),
        "--temperature", "0", "--seed", "42", "--kind", "prefill",
        "--verify-against", "glproc", "--out", out,
    ]
    return run(cmd, cwd=TREE, env={**COMMON_ENV, **ARM_ENV[arm]}, timeout=14400, check=False)

def gate_dispatch(arm, hay):
    contract = last_json_line(hay, "[glcuda-contract]")
    expected = {
        "exact_fusion": True, "defer_ffn_residual": arm == "gwen_wave111",
        "gqa_group": False, "grid2d": True, "r256": False,
        "ntile128": True, "bstage": True, "gemm_n16": True,
        "gemm_n32": False, "gemm_n16_prefetch": True,
        "attn_rows_forced": False, "gqa7_chains": 1,
        "attn_mma4": True, "attn_mma4_regq": True, "attn_mma4_av": True,
    }
    bad = {key: (contract.get(key), value) for key, value in expected.items()
           if contract.get(key) != value}
    attention = last_json_line(hay, "[glcuda-attn]")
    gemm = json_lines(hay, "[glcuda-gemm]")
    if bad or attention.get("path") != "mma4-regq-avmma" or attention.get("ntok") != 244:
        raise RuntimeError(f"{arm} dispatch mismatch: bad={bad}, attention={attention}")
    expected_paths = {"bstage-n16-prefetch", "bstage-n16-m32"}
    if {row.get("path") for row in gemm} != expected_paths or any(row.get("ntok") != 244 for row in gemm):
        raise RuntimeError(f"{arm} GEMM dispatch mismatch: {gemm}")
    return {"contract": contract, "attention": attention, "gemm": gemm}

def gwen_stats(path):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    engine, workload = data.get("engine") or {}, data.get("workload") or {}
    if engine.get("name") != "glcuda" or engine.get("backend") != "cuda" or not engine.get("available"):
        raise RuntimeError(f"wrong GwenLand engine: {engine}")
    expected = {
        "engine": "glcuda", "kind": "prefill", "prompt": FIXED_PROMPT,
        "seed": 42, "temperature": 0.0, "max_new_tokens": 1,
        "cold_iters": COLD_ITERS, "warmup_iters": WARMUP_ITERS,
        "measure_iters": MEASURE_ITERS, "verify_against": "glproc",
    }
    bad = {key: (workload.get(key), value) for key, value in expected.items()
           if workload.get(key) != value}
    if bad:
        raise RuntimeError(f"GwenLand workload drift: {bad}")
    validation = data.get("validation") or {}
    parity = [x for x in validation.get("findings", []) if x.get("check") == "parity"]
    match = re.search(r"(\d+)/(\d+) tokens match oracle", parity[-1].get("message", "")) if parity else None
    if validation.get("passed") is not True or not match or tuple(map(int, match.groups())) != (50, 50):
        raise RuntimeError(f"GwenLand full oracle failed: {validation}")
    rows = (data.get("measurements") or {}).get("iterations") or []
    cold = (data.get("measurements") or {}).get("cold") or []
    if len(rows) != MEASURE_ITERS or len(cold) != COLD_ITERS:
        raise RuntimeError(f"GwenLand sample-count drift: measured={len(rows)}, cold={len(cold)}")
    counts = [int(x.get("prompt_tokens", 0)) for x in rows]
    latency_ms = [float(x.get("prefill_ms", 0)) for x in rows]
    if len(set(counts)) != 1 or counts[0] != 244 or not all(math.isfinite(x) and x > 0 for x in latency_ms):
        raise RuntimeError(f"GwenLand timing contract failed: counts={counts}, latency={latency_ms}")
    tps = [244000.0 / x for x in latency_ms]
    return {
        "p50_tps": percentile(tps, .5), "p90_tps": percentile(tps, .9),
        "p99_tps": percentile(tps, .99), "p50_ms": percentile(latency_ms, .5),
        "p90_ms": percentile(latency_ms, .9), "p99_ms": percentile(latency_ms, .99),
        "max_ms": max(latency_ms), "samples_tps": tps, "samples_ms": latency_ms,
        "oracle": "50/50 tokens match oracle", "raw_validation": validation,
    }

def run_llamacpp(tag):
    cmd = [LLAMA_BENCH, "-m", MODEL_PATH, "-p", "244", "-n", "0",
           "-r", str(MEASURE_ITERS), "-ngl", "99", "-o", "json"]
    rc, out, err = sh(cmd, timeout=14400, env=LB_ENV)
    save_command_log(f"llamabench-{tag}.log", cmd, rc, out, err)
    if rc:
        raise RuntimeError(f"llama-bench failed at {tag}")
    payload = json.loads(out)
    rows = [row for row in payload if int(row.get("n_prompt", 0)) == 244 and int(row.get("n_gen", -1)) == 0]
    if len(rows) != 1:
        raise RuntimeError(f"llama-bench row mismatch: {payload}")
    row = rows[0]
    samples_tps = [float(x) for x in row.get("samples_ts", [])]
    samples_ns = [int(x) for x in row.get("samples_ns", [])]
    if (len(samples_tps) != MEASURE_ITERS or len(samples_ns) != MEASURE_ITERS or
            row.get("backends") != "CUDA" or row.get("gpu_info") != "Tesla T4" or
            int(row.get("n_gpu_layers", 0)) != 99 or
            not str(row.get("build_commit", "")).startswith(LLAMA_COMMIT[:7])):
        raise RuntimeError(f"llama-bench contract failed: {row}")
    latency_ms = [x / 1_000_000.0 for x in samples_ns]
    if not all(math.isfinite(x) and x > 0 for x in samples_tps + latency_ms):
        raise RuntimeError(f"llama-bench invalid samples: {row}")
    return {
        "p50_tps": percentile(samples_tps, .5), "p90_tps": percentile(samples_tps, .9),
        "p99_tps": percentile(samples_tps, .99), "p50_ms": percentile(latency_ms, .5),
        "p90_ms": percentile(latency_ms, .9), "p99_ms": percentile(latency_ms, .99),
        "max_ms": max(latency_ms), "samples_tps": samples_tps,
        "samples_ms": latency_ms, "raw": row,
    }

DISPATCH = {}
PROMPT_TOKENS = None
for arm in ARM_ENV:
    probe_path = RESULTS / f"{arm}-probe.json"
    p = run_gwenland(arm, probe_path, 0, 1, 1)
    save_log(f"{arm}-probe.log", p)
    if p.returncode:
        fail_with_archive("h2h-probe", f"{arm} production probe failed")
    DISPATCH[arm] = gate_dispatch(arm, p.stdout + "\n" + p.stderr)
    probe_data = json.loads(probe_path.read_text(encoding="utf-8"))
    count = int(probe_data["measurements"]["iterations"][0]["prompt_tokens"])
    if count != 244:
        fail_with_archive("h2h-probe", f"{arm} prompt contract drift: {count} != 244")
    PROMPT_TOKENS = count

# Exact effective-prompt gate. glbench's Runtime wraps chat models with this
# single-turn ChatML sequence. Ask glcore for those ids, ask the pinned
# llama-tokenize tool for ids from the byte-identical formatted prompt, then
# require the complete arrays to match before measuring either engine.
raw_prompt_path = RESULTS / "raw-user-prompt.txt"
chatml_prompt_path = RESULTS / "effective-chatml-prompt.txt"
raw_prompt_path.write_text(FIXED_PROMPT, encoding="utf-8")
chatml_prompt_path.write_text(CHATML_PROMPT, encoding="utf-8")

helper_path = TREE / "glcore/examples/wave113_tokenizer_ids.rs"
helper_path.write_text(r"""use glcore::tokenizer::GllmTokenizer;
use std::{env, fs, io};

fn main() -> Result<(), Box<dyn std::error::Error>> {
    let args: Vec<String> = env::args().collect();
    if args.len() != 3 {
        return Err(io::Error::other("usage: helper MODEL.gguf PROMPT.txt").into());
    }
    let prompt = fs::read_to_string(&args[2])?;
    let tokenizer = GllmTokenizer::from_gguf_path(&args[1])?;
    let ids = tokenizer
        .encode_chat(&prompt)?
        .ok_or_else(|| io::Error::other("model has no ChatML markers"))?;
    let joined = ids.iter().map(u32::to_string).collect::<Vec<_>>().join(",");
    println!("GLCORE_CHAT_IDS=[{joined}]");
    Ok(())
}
""", encoding="utf-8")
helper_cmd = [
    CARGO, "run", "--release", "-p", "glcore", "--example",
    "wave113_tokenizer_ids", "--locked", "--", MODEL_PATH, raw_prompt_path,
]
p = run(
    helper_cmd, cwd=TREE, env={"CARGO_TARGET_DIR": str(TARGET)},
    timeout=1800, check=False,
)
save_log("glcore-exact-prompt-ids.log", p)
glcore_match = re.search(r"GLCORE_CHAT_IDS=(\[[0-9,]+\])", p.stdout)
if p.returncode or not glcore_match:
    fail_with_archive(
        "glcore-tokenizer-exact",
        f"cannot collect glcore ChatML ids: rc={p.returncode}",
    )
glcore_ids = json.loads(glcore_match.group(1))

tokenize_cmd = [
    LLAMA_TOKENIZE, "-m", MODEL_PATH, "-f", chatml_prompt_path,
    # LLAMA_EXAMPLE_TOKENIZE enables parse_special by default at LLAMA_PIN.
    # That build exposes only --no-parse-special for this tool; --parse-special
    # belongs to imatrix and is intentionally not accepted by llama-tokenize.
    "--no-escape", "--no-bos", "--ids", "--show-count",
]
rc, tokenize_out, tokenize_err = sh(tokenize_cmd, timeout=300)
save_command_log(
    "llama-tokenize-exact-prompt.log", tokenize_cmd,
    rc, tokenize_out, tokenize_err,
)
tokenize_text = tokenize_out + "\n" + tokenize_err
ids_match = re.search(r"(?m)^(\[[0-9, ]+\])\s*$", tokenize_out)
token_matches = re.findall(r"Total number of tokens:\s*(\d+)", tokenize_text)
if rc or not ids_match or not token_matches:
    tail = tokenize_text[-1000:].replace("\n", " | ")
    fail_with_archive(
        "llama-tokenize-exact",
        f"tokenizer gate failed: rc={rc}, tail={tail}",
    )
llama_ids = json.loads(ids_match.group(1))
first_mismatch = next(
    (i for i, pair in enumerate(zip(glcore_ids, llama_ids)) if pair[0] != pair[1]),
    None,
)
TOKENIZER_EXACT = {
    "prompt_tokens": int(token_matches[-1]),
    "glcore_tokens": len(glcore_ids),
    "llama_tokens": len(llama_ids),
    "ids_exact": glcore_ids == llama_ids,
    "first_mismatch": first_mismatch,
    "method": "glcore encode_chat vs llama-tokenize ChatML --ids",
}
if (TOKENIZER_EXACT["prompt_tokens"] != PROMPT_TOKENS or
        len(glcore_ids) != PROMPT_TOKENS or glcore_ids != llama_ids):
    fail_with_archive(
        "tokenizer-exact",
        f"effective prompt token mismatch: {TOKENIZER_EXACT}",
    )
print("exact-text tokenizer cross-check:", TOKENIZER_EXACT)

RECORDS = []
try:
    for repeat, order in enumerate(ORDERS):
        for position, arm in enumerate(order):
            if arm in ARM_ENV:
                out_path = RESULTS / f"{arm}-r{repeat}-p{position}.json"
                p = run_gwenland(arm, out_path, COLD_ITERS, WARMUP_ITERS, MEASURE_ITERS)
                save_log(f"{arm}-r{repeat}-p{position}.log", p)
                if p.returncode:
                    raise RuntimeError(f"{arm} failed at repeat {repeat}")
                gate_dispatch(arm, p.stdout + "\n" + p.stderr)
                stats = gwen_stats(out_path)
            else:
                stats = run_llamacpp(f"r{repeat}-p{position}")
                (RESULTS / f"llamacpp-r{repeat}-p{position}.json").write_text(
                    json.dumps(stats["raw"], indent=2), encoding="utf-8")
            RECORDS.append({"repeat": repeat, "position": position, "arm": arm, **stats})
            print(f"{arm:14s} r{repeat} p{position}: {stats['p50_tps']:8.1f} tok/s | "
                  f"{stats['p50_ms']:.3f}/{stats['p90_ms']:.3f}/{stats['p99_ms']:.3f} ms", flush=True)
except Exception:
    (RESULTS / "h2h-partial.json").write_text(json.dumps(RECORDS, indent=2), encoding="utf-8")
    fail_with_archive("h2h-production", traceback.format_exc())

SUMMARY = {}
for arm in (*ARM_ENV, "llamacpp"):
    rows = [row for row in RECORDS if row["arm"] == arm]
    SUMMARY[arm] = {
        "p50_tps": statistics.median(row["p50_tps"] for row in rows),
        "p90_tps": statistics.median(row["p90_tps"] for row in rows),
        "p99_tps": statistics.median(row["p99_tps"] for row in rows),
        "p50_ms": statistics.median(row["p50_ms"] for row in rows),
        "p90_ms": statistics.median(row["p90_ms"] for row in rows),
        "p99_ms": statistics.median(row["p99_ms"] for row in rows),
        "max_ms": statistics.median(row["max_ms"] for row in rows),
        "min_session_p50_tps": min(row["p50_tps"] for row in rows),
        "max_session_p50_tps": max(row["p50_tps"] for row in rows),
        "positions": [row["position"] for row in rows], "sessions": len(rows),
    }

PAIRED = []
for repeat in range(PRODUCTION_REPEATS):
    rows = {row["arm"]: row for row in RECORDS if row["repeat"] == repeat}
    PAIRED.append({
        "repeat": repeat,
        "wave111_over_retained": rows["gwen_wave111"]["p50_tps"] / rows["gwen_retained"]["p50_tps"] - 1.0,
        "wave111_over_llamacpp": rows["gwen_wave111"]["p50_tps"] / rows["llamacpp"]["p50_tps"] - 1.0,
        "wave111_minus_llamacpp_tps": rows["gwen_wave111"]["p50_tps"] - rows["llamacpp"]["p50_tps"],
    })

retained = SUMMARY["gwen_retained"]
candidate = SUMMARY["gwen_wave111"]
llama = SUMMARY["llamacpp"]
internal_delta = candidate["p50_tps"] / retained["p50_tps"] - 1.0
h2h_delta = candidate["p50_tps"] / llama["p50_tps"] - 1.0
all_positive = all(row["wave111_over_retained"] > 0 for row in PAIRED)
tail_delta = candidate["max_ms"] / retained["max_ms"] - 1.0
retention_ok = internal_delta > 0 and all_positive and tail_delta <= 0.05
target_reached = candidate["p50_tps"] >= 15000.0
decision = "GOAL_REACHED" if retention_ok and target_reached else (
    "RETAIN_BELOW_GOAL" if retention_ok else "REJECT_WAVE111_PRODUCTION"
)

RESULT = {
    "wave": 113, "status": "production_measured", "gpu": fields,
    "source_revision": SOURCE_REV, "patch_sha256": PATCH_SHA256,
    "model": MODEL_META, "llama_commit": LLAMA_COMMIT,
    "method": {
        "quant": "Q8_0 vs Q8_0", "prompt_tokens": 244,
        "gwen_oracle": "50/50 every session", "repeats_per_arm": 10,
        "total_sessions": 30, "samples_per_session": 10,
        "cold_iters": COLD_ITERS, "warmup_iters": WARMUP_ITERS,
        "order": ORDERS, "seed": 42, "temperature": 0.0,
        "llama_prompt_note": "llama-bench uses synthetic IDs at the tokenizer-verified 244-token shape",
    },
    "summary": SUMMARY, "paired": PAIRED,
    "comparison": {
        "wave111_over_retained": internal_delta,
        "wave111_over_llamacpp": h2h_delta,
        "wave111_minus_llamacpp_tps": candidate["p50_tps"] - llama["p50_tps"],
        "all_internal_pairs_positive": all_positive,
        "tail_max_delta": tail_delta, "retention_ok": retention_ok,
    },
    "tokenizer_exact": TOKENIZER_EXACT, "dispatch": DISPATCH,
    "decision": decision,
    "target_15000_tps_achieved": retention_ok and target_reached,
}
(RESULTS / "production-records.json").write_text(json.dumps(RECORDS, indent=2), encoding="utf-8")
(RESULTS / "wave113-h2h-q8.json").write_text(json.dumps(RESULT, indent=2), encoding="utf-8")
(RESULTS / "PRODUCTION_SUCCESS.json").write_text(
    json.dumps({"status": "valid", "decision": decision}, indent=2), encoding="utf-8")
archive()
print("WAVE113_RESULT", json.dumps(RESULT, indent=2), flush=True)
